<a href="https://colab.research.google.com/github/ashishkunche9-lab/blockchain-projects/blob/main/04-digital-signatures/digital_signatures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json, time, random, copy
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
from cryptography.exceptions import InvalidSignature

# Each operator has a private key (secret) and a public key (shared)
private_keys, public_keys = {}, {}
for name in ["Ravi", "Sita", "Kiran"]:
    k = Ed25519PrivateKey.generate()
    private_keys[name] = k
    public_keys[name] = k.public_key()

def payload(rec):
    data = {k: rec[k] for k in ("id", "operator", "alarm", "value", "time")}
    return json.dumps(data, sort_keys=True).encode()

def sign_record(operator, rec_id, alarm, value):
    rec = {"id": rec_id, "operator": operator, "alarm": alarm, "value": value,
           "time": time.strftime("%Y-%m-%d %H:%M:%S")}
    rec["signature"] = private_keys[operator].sign(payload(rec)).hex()
    return rec

def verify_record(rec):
    pub = public_keys.get(rec["operator"])
    if pub is None:
        return "UNKNOWN OPERATOR"
    try:
        pub.verify(bytes.fromhex(rec["signature"]), payload(rec))
        return "VALID"
    except InvalidSignature:
        return "INVALID"

# Simulated turbine alarms, each signed by the operator who acknowledged it
random.seed(1)
alarms = ["Vibration (mm/s)", "Bearing temp (C)", "Lube oil pressure (bar)"]
records = [sign_record(random.choice(list(private_keys)), i,
                       random.choice(alarms), round(random.uniform(1, 95), 1))
           for i in range(6)]

print(len(records), "signed records")
print("All valid:", all(verify_record(r) == "VALID" for r in records))

6 signed records
All valid: True


In [2]:
import pandas as pd
from IPython.display import display

def table(recs):
    rows = [{"#": r["id"], "Operator": r["operator"], "Alarm": r["alarm"],
             "Value": r["value"], "Signature": r["signature"][:10] + "...",
             "Status": verify_record(r)} for r in recs]
    return pd.DataFrame(rows)

def color(row):
    c = "#c8f7c5" if row["Status"] == "VALID" else "#f7a5a5"
    return [f"background-color: {c}; color: black"] * len(row)

print("STEP 1: Original signed records")
display(table(records).style.apply(color, axis=1).hide(axis="index"))

tampered = copy.deepcopy(records)
tampered[2]["value"] = 0.5
print("\nSTEP 2: Someone edits record 2")
display(table(tampered).style.apply(color, axis=1).hide(axis="index"))

STEP 1: Original signed records


#,Operator,Alarm,Value,Signature,Status
0,Ravi,Lube oil pressure (bar),80.700000,174a2eb16a...,VALID
1,Ravi,Bearing temp (C),12.100000,b2d6cb1286...,VALID
2,Sita,Bearing temp (C),62.200000,65d94d9cfb...,VALID
3,Ravi,Vibration (mm/s),46.900000,7efefb9063...,VALID
4,Sita,Bearing temp (C),58.100000,a4a56d3fab...,VALID
5,Ravi,Lube oil pressure (bar),42.900000,0899901665...,VALID



STEP 2: Someone edits record 2


#,Operator,Alarm,Value,Signature,Status
0,Ravi,Lube oil pressure (bar),80.700000,174a2eb16a...,VALID
1,Ravi,Bearing temp (C),12.100000,b2d6cb1286...,VALID
2,Sita,Bearing temp (C),0.500000,65d94d9cfb...,INVALID
3,Ravi,Vibration (mm/s),46.900000,7efefb9063...,VALID
4,Sita,Bearing temp (C),58.100000,a4a56d3fab...,VALID
5,Ravi,Lube oil pressure (bar),42.900000,0899901665...,VALID


In [3]:
base = records[1]
attacker_key = Ed25519PrivateKey.generate()

def run(name, change):
    rec = copy.deepcopy(base)
    change(rec)
    print(f"{name:36} -> {verify_record(rec)}")

def edit_value(r): r["value"] = 0.5
def edit_alarm(r): r["alarm"] = "Normal"
def blame_other(r): r["operator"] = "Sita" if r["operator"] != "Sita" else "Ravi"
def forge(r):
    r["operator"] = "Ravi"; r["value"] = 0.5
    r["signature"] = attacker_key.sign(payload(r)).hex()
def unknown(r): r["operator"] = "Mallory"
def replay(r): r["signature"] = records[3]["signature"]
def stolen_key(r):
    r["value"] = 0.5
    r["signature"] = private_keys[r["operator"]].sign(payload(r)).hex()

run("No change", lambda r: None)
run("Edit a value", edit_value)
run("Edit alarm name", edit_alarm)
run("Blame a different operator", blame_other)
run("Forge as Ravi (attacker's own key)", forge)
run("Unknown operator", unknown)
run("Copy signature from another record", replay)
run("Stolen private key", stolen_key)

No change                            -> VALID
Edit a value                         -> INVALID
Edit alarm name                      -> INVALID
Blame a different operator           -> INVALID
Forge as Ravi (attacker's own key)   -> INVALID
Unknown operator                     -> UNKNOWN OPERATOR
Copy signature from another record   -> INVALID
Stolen private key                   -> VALID
